## Dataset Completeness and Current-Consistency Assessment

The first code cell evaluates the completeness and internal consistency of BLDC motor measurements acquired with the laboratory power supply.

1. **File enumeration**  
   Files matching `analize_*.csv` are enumerated from the `data` directory.

2. **Metadata extraction**  
   A regular expression parses the motor condition, experiment identifier, optional quality label, rotational-speed setpoint, load-current setting, and power-source suffix from each filename. The experiment identifier is decoded into a zero-based motor index.

3. **Analytical subset**  
   The analysis includes clean recordings with the `mait` suffix, denoting the UNIT-T UTP3305 laboratory power supply. Recordings labelled `ENV`, `SF`, or `bat` are excluded. The switched-off baseline recording (`analize_0rpm_0mA.csv`), filenames that do not match the naming convention, and combinations outside the configured motor-condition-speed-load domain are also excluded.

4. **Reference measurement domain**  
   Motor 0 is assigned to the `demag` condition. Motors 1–3 are assigned to `healthy`, `misalignment`, `front_ball`, and `rear_ball`. The 500, 1000, and 1500 rpm setpoints include load-current settings of 19, 38, 64, 96, and 128 mA. The supplementary 2000, 2500, and 3000 rpm setpoints include 96 and 128 mA.

5. **Current aggregation**  
   Files and non-missing values from the `curr_raw` column are grouped by `(motor, state, rpm, mA)`. The arithmetic mean is calculated from the accumulated sum and sample count for each group, and the matching file count is retained for each combination.

6. **Completeness assessment**  
   All configured `(motor, state, rpm, mA)` combinations are generated. A combination is classified as present when at least one valid `curr_raw` sample is available and as missing otherwise.

7. **Current-consistency assessment**  
   Within each `(motor, state, rpm)` group, mean current values are ordered by the load-current setting. A decrease relative to the preceding observed setting is recorded as a monotonicity violation.

8. **Output**  
   Separate tables are produced for each motor. Each table includes the number of files matching the given combination. Missing combinations and monotonicity violations are highlighted in red. The report concludes with the total numbers of missing combinations and detected violations.

In [66]:
from pathlib import Path
import importlib
import pandas as pd
from IPython.display import Markdown, display
import validation_utils as vu

DATA_DIR = Path("../data")
vu = importlib.reload(vu)
MAIT_CURRENT_GRID = {
    500: [19, 38, 64, 96, 128],
    1000: [19, 38, 64, 96, 128],
    1500: [19, 38, 64, 96, 128],
    2000: [96, 128],
    2500: [96, 128],
    3000: [96, 128],
}
MAIT_EXPECTED_STATES = {
    # 0: ["demag"],
    1: ["healthy", "misalignment", "front_ball", "rear_ball"],
    2: ["healthy", "misalignment", "front_ball", "rear_ball"],
    3: ["healthy", "misalignment", "front_ball", "rear_ball"],
}
avg_df, read_errors = vu.build_current_validation_df(
    DATA_DIR,
    source="mait",
    expected_states=MAIT_EXPECTED_STATES,
    current_grid=MAIT_CURRENT_GRID,
)
vu.render_current_validation_report(
    avg_df,
    read_errors,
    title="Laboratory power supply only",
    expected_states=MAIT_EXPECTED_STATES,
)

## Laboratory power supply only


Motor 1


motor,state,rpm,mA,file_count,avg_current,current_decreased_vs_previous_ma
1,healthy,500,19,1,167.699676,False
1,healthy,500,38,1,172.669736,False
1,healthy,500,64,1,183.549170,False
1,healthy,500,96,1,283.427764,False
1,healthy,500,128,1,805.664332,False
1,healthy,1000,19,1,271.599124,False
1,healthy,1000,38,1,276.182557,False
1,healthy,1000,64,2,298.028057,False
1,healthy,1000,96,2,475.132935,False
1,healthy,1000,128,1,1253.589920,False



Motor 2


motor,state,rpm,mA,file_count,avg_current,current_decreased_vs_previous_ma
2,healthy,500,19,1,167.114967,False
2,healthy,500,38,1,169.887703,False
2,healthy,500,64,1,180.842709,False
2,healthy,500,96,1,284.954564,False
2,healthy,500,128,1,711.866055,False
2,healthy,1000,19,1,253.015385,False
2,healthy,1000,38,1,261.034025,False
2,healthy,1000,64,1,286.345845,False
2,healthy,1000,96,1,473.741580,False
2,healthy,1000,128,1,1221.040356,False



Motor 3


motor,state,rpm,mA,file_count,avg_current,current_decreased_vs_previous_ma
3,healthy,500,19,1,146.158394,False
3,healthy,500,38,1,150.066230,False
3,healthy,500,64,1,162.810514,False
3,healthy,500,96,1,270.802671,False
3,healthy,500,128,1,716.680159,False
3,healthy,1000,19,1,226.111852,False
3,healthy,1000,38,1,232.972703,False
3,healthy,1000,64,1,260.593809,False
3,healthy,1000,96,1,486.165582,False
3,healthy,1000,128,1,1201.699612,False



Missing combinations: 0
Current decrease flags found: 0


In [67]:
state_progress_df = (
    avg_df.groupby("state", sort=False)
    .agg(
        present_combinations=("is_missing", lambda s: int((~s).sum())),
        expected_combinations=("is_missing", "size"),
        missing_combinations=("is_missing", "sum"),
        file_count=("file_count", "sum"),
    )
    .reset_index()
)

state_progress_df["progress_pct"] = (
    state_progress_df["present_combinations"]
    / state_progress_df["expected_combinations"]
    * 100
)

state_progress_df["progress_pct"] = state_progress_df["progress_pct"].round(1)
state_progress_df["progress_pct"] = state_progress_df["progress_pct"].map(lambda v: f"{v:.1f}%")

display(Markdown("## State completion progress ''mait''"))
display(
    state_progress_df[
        [
            "state",
            "present_combinations",
            "expected_combinations",
            "missing_combinations",
            "file_count",
            "progress_pct",
        ]
    ]
)

## State completion progress ''mait''

,state,present_combinations,expected_combinations,missing_combinations,file_count,progress_pct
0,healthy,63,63,0,69,100.0%
1,misalignment,63,63,0,63,100.0%
2,front_ball,63,63,0,63,100.0%
3,rear_ball,63,63,0,63,100.0%


In [60]:
BAT_CURRENT_GRID = {
    500: [19, 38, 64, 96, 128],
    1000: [19, 38, 64, 96, 128],
    1500: [19, 38, 64, 96, 128],
    # 2000: [96, 128],
    # 2500: [96, 128],
    # 3000: [96, 128],
}
BAT_EXPECTED_STATES = {
    # 0: ["demag"],
    1: ["healthy", "misalignment", "front_ball", "rear_ball"],
    2: ["healthy", "misalignment", "front_ball", "rear_ball"],
    3: ["healthy", "misalignment", "front_ball", "rear_ball"],
}
avg_df_bat, read_errors_bat = vu.build_current_validation_df(
    DATA_DIR,
    source="bat",
    expected_states=BAT_EXPECTED_STATES,
    current_grid=BAT_CURRENT_GRID,
)
vu.render_current_validation_report(
    avg_df_bat,
    read_errors_bat,
    title="Battery supply only",
    expected_states=BAT_EXPECTED_STATES,
)

## Battery supply only


Motor 1


motor,state,rpm,mA,file_count,avg_current,current_decreased_vs_previous_ma
1,healthy,500,19,1,170.395572,False
1,healthy,500,38,1,174.490142,False
1,healthy,500,64,1,188.350833,False
1,healthy,500,96,1,303.606100,False
1,healthy,500,128,1,683.265509,False
1,healthy,1000,19,1,254.154060,False
1,healthy,1000,38,1,263.397701,False
1,healthy,1000,64,1,283.454489,False
1,healthy,1000,96,1,453.976228,False
1,healthy,1000,128,1,1173.864335,False



Motor 2


motor,state,rpm,mA,file_count,avg_current,current_decreased_vs_previous_ma
2,healthy,500,19,0,-,-
2,healthy,500,38,0,-,-
2,healthy,500,64,0,-,-
2,healthy,500,96,0,-,-
2,healthy,500,128,0,-,-
2,healthy,1000,19,0,-,-
2,healthy,1000,38,0,-,-
2,healthy,1000,64,0,-,-
2,healthy,1000,96,0,-,-
2,healthy,1000,128,0,-,-



Motor 3


motor,state,rpm,mA,file_count,avg_current,current_decreased_vs_previous_ma
3,healthy,500,19,0,-,-
3,healthy,500,38,0,-,-
3,healthy,500,64,0,-,-
3,healthy,500,96,0,-,-
3,healthy,500,128,0,-,-
3,healthy,1000,19,0,-,-
3,healthy,1000,38,0,-,-
3,healthy,1000,64,0,-,-
3,healthy,1000,96,0,-,-
3,healthy,1000,128,0,-,-



Missing combinations: 156
Current decrease flags found: 0
